# Practical 11 — Reinforcement Learning using Q-Learning

Train an agent in a small Grid World using epsilon-greedy exploration and exploitation.

## 1. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 2. Define Grid World, states and actions

In [ ]:
ROWS, COLS = 4, 4
START, GOAL = (0, 0), (3, 3)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
ACTION_NAMES = ["↑", "↓", "←", "→"]
n_states, n_actions = ROWS * COLS, len(ACTIONS)


## 3. Create transition function

In [ ]:
def state_id(state):
    return state[0] * COLS + state[1]

def step(state, action_index):
    dr, dc = ACTIONS[action_index]
    nr, nc = state[0] + dr, state[1] + dc
    if not (0 <= nr < ROWS and 0 <= nc < COLS):
        return state, -5, False
    next_state = (nr, nc)
    if next_state == GOAL:
        return next_state, 20, True
    return next_state, -1, False


## 4. Train Q-learning agent

In [ ]:
q_table = np.zeros((n_states, n_actions))
episodes = 3000
alpha, gamma = 0.1, 0.95
epsilon, epsilon_min, epsilon_decay = 1.0, 0.05, 0.995
episode_rewards = []
for episode in range(episodes):
    state, total_reward = START, 0
    for _ in range(100):
        s = state_id(state)
        if np.random.random() < epsilon:
            action = np.random.randint(n_actions)  # exploration
        else:
            action = int(np.argmax(q_table[s]))    # exploitation
        next_state, reward, done = step(state, action)
        ns = state_id(next_state)
        target = reward if done else reward + gamma * np.max(q_table[ns])
        q_table[s, action] += alpha * (target - q_table[s, action])
        state = next_state
        total_reward += reward
        if done:
            break
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)
print("Final epsilon:", round(epsilon, 3))


## 5. Plot learning progress

In [ ]:
window = 100
average_reward = np.convolve(episode_rewards, np.ones(window) / window, mode="valid")
plt.plot(average_reward)
plt.xlabel("Episode")
plt.ylabel("Average reward (100-episode window)")
plt.title("Q-learning training progress")
plt.show()


## 6. Display learned policy

In [ ]:
for r in range(ROWS):
    row = []
    for c in range(COLS):
        state = (r, c)
        row.append("G" if state == GOAL else ACTION_NAMES[int(np.argmax(q_table[state_id(state)]))])
    print(" ".join(row))


## 7. Test the learned policy

In [ ]:
state = START
path = [state]
for _ in range(ROWS * COLS * 2):
    if state == GOAL:
        break
    action = int(np.argmax(q_table[state_id(state)]))
    state, reward, done = step(state, action)
    path.append(state)
    if done:
        break
print("Learned path:", path)
print("Reached goal:", state == GOAL)
